<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_41_auth_security/note_lesson_41_auth_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔐 Урок 41 — Автентифікація й безпека: рефакторинг 4 застосунку нотаток

Застосунок нотаток уже вміє «кожен бачить лише свої нотатки». Сьогодні нотатка може належати **групі** («Сім'я»), паролі розбираємо до байта, API отримує **JWT**, а налаштування — перевірку перед сервером. На кожному кроці — один і той самий набір героїв: **Олена** (авторка нотаток), **Тарас** (у її групі «Сім'я»), **Іван** (сторонній).

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | проєкт у ноутбуці, налаштування автентифікації | 0.1 | |
| 1 | нотатки групи: хто бачить, хто змінює; сайт і API — два входи | 1.1–1.7 | **Задача 1** |
| 2 | паролі: формат рядка, сіль, PBKDF2 вручну, ціна перебору, валідатори, скидання пароля | 2.1–2.8 | |
| 3 | JWT: розібрати, підписати вручну, підробки, refresh, порядок автентифікаторів, throttle | 3.1–3.10 | **Задача 2** |
| 4 | налаштування безпеки: `check --deploy`, заголовки й cookie, ключ з репозиторію | 4.1–4.5 | |
| 5 | OWASP: ін'єкція проти ORM | 5.1 | |
| 6 | правило автора в одному місці — DRF-permission | 6.1–6.2 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, `class`, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Хеші, солі й токени у тебе будуть інші (вони випадкові) — важливі **статуси й висновки**, а не самі рядки.
- Інтернет і сервер не потрібні: база — SQLite, запити — тестовий клієнт Django/DRF.
- Зміни коду, знахідки й архітектура — у книзі курсу: [Урок 41](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_41/); теорія — частина VII Django-книги [Auth і безпека](https://nikoriakviktor.github.io/notes_chat_app/07_auth_and_security/); паролі й JWT без Django — частина VIII FastAPI-книги [Безпека API](https://nikoriakviktor.github.io/news_hub/08_security/).

---
## 0. Проєкт у ноутбуці

`crispy_notes_project` (сторінки + DRF API) + стартовий код автентифікації (групи, скидання пароля, налаштування безпеки) + JWT (`/api/token/`). У Colab клітинка нижче завантажить лише папку уроку й встановить залежності; локально — просто перейде в папку проєкту. База щоразу створюється заново.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_41_auth_security"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("crispy_notes_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import crispy_bootstrap5, debug_toolbar, drf_spectacular, rest_framework, rest_framework_simplejwt  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crispy_notes_project/requirements.txt"], check=True)

os.chdir("crispy_notes_project")
if os.path.exists("db.sqlite3"):
    os.remove("db.sqlite3")                             # щоразу — чиста база
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

Наступна клітинка підключає Django до ноутбука і створює трьох користувачів з **однаковим** паролем (пізніше це знадобиться) та дві нотатки Олени. Логер `django.request` приглушено: інакше кожна відповідь `4xx` друкує рядок `Forbidden: /api/…` — статуси ми друкуємо самі.

In [ ]:
import base64
import json
import logging

import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 34)
django.setup()

from django.conf import settings
from django.contrib.auth.hashers import check_password, make_password
from django.contrib.auth.models import User
from django.core.cache import cache
from django.test import Client
from django.test.utils import setup_test_environment
from rest_framework.test import APIClient

from hello_app import selectors, services
from hello_app.models import Note

setup_test_environment()                               # хост testserver, листи — у mail.outbox
logging.getLogger("django.request").setLevel(logging.ERROR)

olena = User.objects.create_user("olena", email="olena@example.com", password="Sup3r-secret!")
taras = User.objects.create_user("taras", password="Sup3r-secret!")
ivan = User.objects.create_user("ivan", password="Sup3r-secret!")
wifi = services.create_note(user=olena, title="Пароль від Wi-Fi")
gift = services.create_note(user=olena, title="Подарунок Тарасу")
print("користувачі:", [u.username for u in User.objects.all()], "| нотатки Олени:", [wifi.title, gift.title])

### Приклад 0.1. Хто такий «я» для API

Перш ніж змінювати код, подивимось, як API вирішує, **хто** робить запит. Це три налаштування DRF і simplejwt з `settings.py`:

In [ ]:
for name in settings.REST_FRAMEWORK["DEFAULT_AUTHENTICATION_CLASSES"]:
    print("автентифікація:", name)
print("права за замовчуванням:", settings.REST_FRAMEWORK["DEFAULT_PERMISSION_CLASSES"])
print("throttle:", settings.REST_FRAMEWORK["DEFAULT_THROTTLE_RATES"])
print("access живе:", settings.SIMPLE_JWT["ACCESS_TOKEN_LIFETIME"], "| refresh живе:", settings.SIMPLE_JWT["REFRESH_TOKEN_LIFETIME"])

- **Автентифікація** («хто ти») — спершу JWT із заголовка `Authorization: Bearer …`, потім сесія браузера (cookie). Порядок важливий — приклад 3.9.
- **Права** («що тобі можна») за замовчуванням — `IsAuthenticated`: анонім не отримає нічого. Решту прав (чия нотатка) вирішує код уроку.
- Токен доступу живе 5 хвилин, refresh — добу; вхід обмежено 5 спробами за хвилину.

---
## 1. Нотатки групи

Коли нотатку бачить не лише автор, правило «`user=request.user`» розпадається на два:

| Хто | Бачить | Змінює й видаляє |
|---|---|---|
| автор (Олена) | так | так |
| учасник групи нотатки (Тарас) | так | ні — `403` |
| інший користувач (Іван) | ні — `404` | ні — `404` |

### Приклад 1.1. Група «Сім'я» і нотатка групи

Олена створює групу (`services.create_group` додає її саму), додає Тараса і робить `wifi` нотаткою групи. Функція `who_sees(note)` питає селектор `get_user_notes` від імені кожного з трьох — так само, як це робить сторінка `/notes/`.

In [ ]:
PEOPLE = (olena, taras, ivan)


def who_sees(note):
    """Імена тих, у чиєму списку нотаток є note."""
    return [user.username for user in PEOPLE if note in selectors.get_user_notes(user)]

In [ ]:
family = services.create_group(name="Сім'я", creator=olena)
family.user_set.add(taras)
wifi = services.update_note(wifi, group=family)

print("учасники групи:", [u.username for u in family.user_set.all()])
print("Wi-Fi бачать:", who_sees(wifi))
print("Подарунок бачать:", who_sees(gift))
assert who_sees(wifi) == ["olena", "taras"]
assert who_sees(gift) == ["olena"]

Нотатка групи видна обом учасникам, особиста «Подарунок Тарасу» — лише Олені (добре, що Тарас її не бачить). Іван не бачить нічого.

### Приклад 1.2. Один фільтр `OR` у SQL

Як селектор знаходить «свої **або** групи»? `Q(user=user) | Q(group__in=user.groups.all())`. Django перетворює це на один SQL-запит — подивимось на нього (`queryset.query`):

In [ ]:
import inspect

print(inspect.getsource(selectors.get_user_notes).split("\n\n")[0])

In [ ]:
sql = str(selectors.get_user_notes(taras).query)
print(sql[sql.index("WHERE"):sql.index("ORDER BY")])

У `WHERE` — `user_id = …` **OR** `group_id IN (SELECT … з груп користувача)`. Права на читання записані в **одному** місці — у селекторі. І сайт, і API беруть нотатки саме звідси.

### Приклад 1.3. Сайт: хто що бачить

Тестовий клієнт Django `Client` — це «браузер» без мережі: `force_login(user)` ставить cookie сесії, а `get(...)` віддає HTML. Визначимо функцію, яка відкриває список і сторінку нотатки від імені користувача:

In [ ]:
def site_view(user):
    """Що бачить user на сайті: назви з /notes/ і статус сторінки нотатки wifi."""
    browser = Client()
    browser.force_login(user)
    page = browser.get("/notes/").content.decode()
    visible = [title for title in (wifi.title, gift.title) if title in page]
    return visible, browser.get(f"/notes/{wifi.pk}/").status_code

In [ ]:
for user in PEOPLE:
    visible, status = site_view(user)
    print(f"{user.username:6} бачить {visible}; сторінка Wi-Fi → {status}")

Іван отримує `404` на сторінку нотатки: для нього її «немає». Це та сама поведінка, що й у списку, бо обидва беруть дані з селекторів.

### Приклад 1.4. Сайт: учасник групи не видалить чужу нотатку

`note_delete` спершу знаходить нотатку серед видимих, а потім перевіряє `note.user != request.user` — і замість видалення показує повідомлення. Перевіримо: Тарас надсилає форму видалення.

In [ ]:
browser = Client()
browser.force_login(taras)
response = browser.post(f"/notes/{wifi.pk}/delete/", follow=True)
print("куди перенаправили:", response.redirect_chain)
print("повідомлення:", [str(m) for m in response.context["messages"]])
print("нотатка на місці:", Note.objects.filter(pk=wifi.pk).exists())

Сайт захищений: `302` назад до списку з поясненням, нотатка ціла.

### Приклад 1.5. Форма: групу можна вибрати лише зі своїх

Ще один вхід — поле `group` у формі нотатки. Якби там були **всі** групи, Іван міг би підкласти свою нотатку в «Сім'ю» Олени (або підставити id групи вручну). `NoteForm(user=...)` обмежує вибір групами самого користувача:

In [ ]:
from hello_app.forms import NoteForm

for user in PEOPLE:
    choices = [group.name for group in NoteForm(user=user).fields["group"].queryset]
    print(f"{user.username:6} може вибрати групи: {choices}")

In [ ]:
form = NoteForm({"title": "Підкинута", "priority": 1, "group": family.pk}, user=ivan)
print("форма Івана з групою «Сім'я» валідна?", form.is_valid())
print(form.errors["group"])

Чужий id групи форма відкидає з помилкою «вибери правильний варіант»: значення має бути в `queryset` поля. Захист — не в HTML (його можна підробити), а в перевірці на сервері.

**🔮 Прогноз:** тепер Тарас (у групі) і Іван (ні) звертаються до **API** нотатки групи. Які статуси: Тарас `GET`, Тарас `PATCH`, Тарас `DELETE`, Іван `GET`?

<details>
<summary>Відповідь</summary>

`200`, `403`, `403`, `404`. Тарас бачить нотатку, але змінювати може лише автор — чесне «заборонено». Іван нотатку не бачить — для нього її «немає».

</details>

### Задача 1. Статуси API для нотатки групи

Перевір прогноз. Збери словник `codes` з ключами `"taras GET"`, `"taras PATCH"`, `"taras DELETE"`, `"ivan GET"` і статус-кодами відповідей.

- клієнт — `APIClient()`, від імені користувача — `api.force_authenticate(user)` (без пароля й токена, лише для тестів);
- адреса — `url` з наступної клітинки; `PATCH` з тілом `{"title": "Зламано"}` і `format="json"`.

Перевірка — у клітинці після задачі.

In [ ]:
api = APIClient()
url = f"/api/notes/{wifi.pk}/"

In [ ]:
# YOUR CODE HERE


In [ ]:
assert isinstance(codes, dict) and set(codes) == {"taras GET", "taras PATCH", "taras DELETE", "ivan GET"}, \
    "потрібен словник з чотирма ключами: 'taras GET', 'taras PATCH', 'taras DELETE', 'ivan GET'"
print(codes)
assert codes == {"taras GET": 200, "taras PATCH": 403, "taras DELETE": 403, "ivan GET": 404}
assert Note.objects.get(pk=wifi.pk).title == "Пароль від Wi-Fi", "нотатка не мала змінитися"
print("✅ Задача 1 розв'язана")

### Приклад 1.6. Де в API живе перевірка автора

Читання в API бере `_get_note` (селектор: своя або групи; інакше `404`), а зміни — `_get_own_note`, яка додає перевірку автора й кидає `PermissionDenied` (`403`):

In [ ]:
from hello_app.api import NoteViewSet

before_doc, _docstring, after_doc = inspect.getsource(NoteViewSet._get_own_note).split('"""')
print(before_doc.rstrip() + after_doc)                # без docstring — лише код

### Приклад 1.7. Що було б без `_get_own_note`

Знахідка уроку: коли додали групи, селектор почав віддавати й нотатки групи, а сторінки перевіряли автора — API ні. Відтворимо стан «до виправлення»: тимчасово (`mock.patch.object`) підмінимо `_get_own_note` на `_get_note`, тобто приберемо перевірку автора.

**🔮 Прогноз:** що отримає Тарас на `PATCH` у такому API?

<details>
<summary>Відповідь</summary>

`200` — і заголовок нотатки Олени зміниться. Це **Broken Access Control** (OWASP A01): доступ на читання розширили, а один із входів, що пише, забули оновити.

</details>

In [ ]:
from unittest import mock

api = APIClient()
api.force_authenticate(taras)
with mock.patch.object(NoteViewSet, "_get_own_note", NoteViewSet._get_note):
    response = api.patch(url, {"title": "Зламано учасником групи"}, format="json")
print("без перевірки автора: PATCH →", response.status_code, "| заголовок:", Note.objects.get(pk=wifi.pk).title)

print("з перевіркою:         PATCH →", api.patch(url, {"title": "Ще раз"}, format="json").status_code)
wifi = services.update_note(Note.objects.get(pk=wifi.pk), title="Пароль від Wi-Fi")   # повертаємо назву

HTML і API — **два входи** до тих самих даних. Права треба перевіряти на кожному. У розділі 6 це правило винесемо в один клас, щоб не тримати його у двох місцях.

---
## 2. Паролі

Django **ніколи не зберігає пароль**. У полі `user.password` — рядок `алгоритм$ітерації$сіль$хеш`.

### Приклад 2.1. Розбираємо рядок пароля

У Олени й Тараса **однаковий** пароль `Sup3r-secret!`. Функція `parse_password` розкладає рядок з бази на частини:

In [ ]:
def parse_password(encoded):
    """'pbkdf2_sha256$1000000$сіль$хеш' → словник частин."""
    algorithm, iterations, salt, digest = encoded.split("$")
    return {"algorithm": algorithm, "iterations": int(iterations), "salt": salt, "hash": digest}

In [ ]:
olena_parts = parse_password(olena.password)
taras_parts = parse_password(taras.password)
for name, parts in (("olena", olena_parts), ("taras", taras_parts)):
    print(f"{name}: {parts['algorithm']} | {parts['iterations']:,} ітерацій | сіль {parts['salt']} | хеш {parts['hash'][:16]}…")
print("однаковий хеш?", olena_parts["hash"] == taras_parts["hash"])
assert olena_parts["algorithm"] == "pbkdf2_sha256" and olena_parts["iterations"] >= 600_000

**🔮 Прогноз:** чому однаковий пароль дав різні хеші? І як тоді сервер перевіряє пароль при вході?

<details>
<summary>Відповідь</summary>

**Сіль** — випадкова для кожного користувача, і хешується пароль **разом** із сіллю. При вході сервер бере сіль з рядка в базі, рахує хеш введеного пароля з нею й порівнює — це робить `check_password`.

</details>

### Приклад 2.2. `check_password`: перевірити, не знаючи пароля

In [ ]:
print("правильний пароль:     ", check_password("Sup3r-secret!", olena.password))
print("інша перша літера:     ", check_password("sup3r-secret!", olena.password))
print("пароль Олени у Тараса: ", check_password("Sup3r-secret!", taras.password))
print("через модель:          ", olena.check_password("Sup3r-secret!"))

Одна змінена літера — і `False`. Зворотного шляху «хеш → пароль» немає: можна лише порахувати хеш кандидата і порівняти.

### Приклад 2.3. Та сама сіль — той самий хеш

Перевіримо, що різниця справді в солі: `make_password` можна дати сіль явно.

In [ ]:
same_salt_1 = make_password("Sup3r-secret!", salt="FixedSalt123")
same_salt_2 = make_password("Sup3r-secret!", salt="FixedSalt123")
random_salt = make_password("Sup3r-secret!")
print(same_salt_1[-20:], "\n", same_salt_2[-20:], "\n", random_salt[-20:], sep="")
print("однакова сіль → однаковий рядок:", same_salt_1 == same_salt_2)

Хеш — детермінований: та сама сіль і той самий пароль дають той самий результат. Без солі однакові паролі в базі мали б однакові хеші, і зловмисник побачив би, хто має «популярний» пароль, та міг би перевірити заздалегідь пораховані таблиці (rainbow tables). Тому сіль генерується **випадково** для кожного пароля.

### Приклад 2.4. PBKDF2 вручну — без магії

`pbkdf2_sha256` — це стандартна функція `hashlib.pbkdf2_hmac`, яку Django викликає з сіллю і числом ітерацій з того самого рядка. Відтворимо хеш Олени стандартною бібліотекою:

In [ ]:
import hashlib


def pbkdf2_hash(password, salt, iterations):
    """Те саме, що робить PBKDF2PasswordHasher: base64(PBKDF2-HMAC-SHA256(password, salt, iterations))."""
    raw = hashlib.pbkdf2_hmac("sha256", password.encode(), salt.encode(), iterations)
    return base64.b64encode(raw).decode()

In [ ]:
mine = pbkdf2_hash("Sup3r-secret!", olena_parts["salt"], olena_parts["iterations"])
print("порахували:", mine)
print("у базі:    ", olena_parts["hash"])
assert mine == olena_parts["hash"]

Збіглося до символу. Отже, `check_password` — це «розібрати рядок → порахувати PBKDF2 з тією ж сіллю → порівняти». Секретного ключа в цій схемі немає: захищають сіль і **повільність**.

### Приклад 2.5. Ціна перебору

Скільки коштує один хеш? Поміряємо PBKDF2 з налаштованим числом ітерацій і оцінимо перебір усіх 8-літерних паролів з малих латинських літер (26⁸ ≈ 2·10¹¹ варіантів) на одному ядрі.

In [ ]:
import time


def seconds_per_hash(iterations):
    start = time.perf_counter()
    hashlib.pbkdf2_hmac("sha256", b"guess", b"salt", iterations)
    return time.perf_counter() - start

In [ ]:
candidates = 26 ** 8
for iterations in (1_000, 10_000, olena_parts["iterations"]):
    per_hash = seconds_per_hash(iterations)
    years = candidates * per_hash / (3600 * 24 * 365)
    print(f"{iterations:>9,} ітерацій: {per_hash * 1000:8.3f} мс на хеш → перебір ≈ {years:,.1f} років")

Числа залежать від машини, але **співвідношення** — ні: у 1000 разів більше ітерацій — кожна спроба в ~1000 разів дорожча. А відеокарта перебирає в тисячі разів швидше за одне ядро — тому й потрібні мільйон ітерацій **і** довгий пароль. Для входу одна перевірка — непомітно (частка секунди), для перебору бази після витоку — роки. Число ітерацій записане в самому рядку, тож Django збільшує його з кожною версією, а старі паролі перехешовує при наступному вході.

### Приклад 2.6. Валідатори пароля

Повільний хеш не врятує пароль `12345678` — його вгадають першим. `AUTH_PASSWORD_VALIDATORS` у `settings.py` перевіряє новий пароль при реєстрації та зміні. Викличемо їх напряму:

In [ ]:
from django.contrib.auth.password_validation import validate_password
from django.core.exceptions import ValidationError


def password_problems(password, user=None):
    """Список повідомлень валідаторів; порожній — пароль прийнято."""
    try:
        validate_password(password, user=user)
    except ValidationError as error:
        return error.messages
    return []

In [ ]:
for candidate in ("12345678", "qwerty", "olena2024", "Sup3r-secret!"):
    print(f"{candidate:14} → {password_problems(candidate, user=olena) or 'прийнято'}")

Чотири валідатори: схожість з ім'ям/email користувача, мінімальна довжина, список поширених паролів, «лише цифри». Повідомлення — українською, бо `LANGUAGE_CODE = "uk"`.

### Приклад 2.7. Скидання пароля: однакова відповідь для всіх

`/accounts/password_reset/` надсилає лист з посиланням. Тестове оточення кладе листи в `mail.outbox` замість пошти. Надішлемо форму для відомого й невідомого email:

In [ ]:
from django.core import mail

mail.outbox.clear()
for email in ("olena@example.com", "nobody@example.com"):
    response = Client().post("/accounts/password_reset/", {"email": email})
    print(f"{email:20} → {response.status_code} {response['Location']}")
print("листів:", len(mail.outbox), "| кому:", mail.outbox[0].to)
link = next(line for line in mail.outbox[0].body.splitlines() if "/accounts/reset/" in line)
print("посилання:", link.strip().rsplit("/", 3)[0] + "/…/")

Відповідь однакова (`302` на «лист надіслано»), а лист пішов лише Олені. Інакше форма скидання стала б способом **перевіряти, хто зареєстрований** (user enumeration, OWASP A07).

### Приклад 2.8. Посилання скидання — одноразове

Токен у посиланні робить `default_token_generator`: він підписаний `SECRET_KEY` і залежить від **хешу пароля** та часу останнього входу. Що станеться з токеном, коли пароль зміниться — навіть на той самий?

In [ ]:
from django.contrib.auth.tokens import default_token_generator

reset_token = default_token_generator.make_token(olena)
print("токен:", reset_token)
print("дійсний до зміни пароля:", default_token_generator.check_token(olena, reset_token))

olena.set_password("Sup3r-secret!")        # той самий пароль, але нова сіль → новий рядок у базі
olena.save()
print("дійсний після зміни:    ", default_token_generator.check_token(olena, reset_token))

Пароль той самий, але сіль нова — отже, новий хеш і старе посилання більше не діє. Тому посилання з листа працює рівно один раз: після встановлення нового пароля воно «згорає» саме, без таблиці використаних токенів.

---
## 3. JWT

Сесія (cookie + CSRF) зручна браузеру. Мобільному застосунку, скрипту чи боту потрібне інше: отримати **токен** і надсилати його в заголовку `Authorization: Bearer …`.

| | Сесія (cookie) | JWT (Bearer) |
|---|---|---|
| Де «я увійшов» | ключ у cookie, дані — на сервері | усе в самому токені, підписаному `SECRET_KEY` |
| Хто надсилає | браузер сам, з кожним запитом | клієнт явно, в заголовку |
| Вийти / відкликати | видалити сесію на сервері | токен діє до `exp`; тому access-токен короткий |

### Приклад 3.1. Без токена — `401` і підказка

Лічильник спроб входу живе в кеші, тож перед кожним прикладом з `/api/token/` його скидаємо.

In [ ]:
cache.clear()
api = APIClient()
response = api.get("/api/notes/")
print(response.status_code, "| WWW-Authenticate:", response["WWW-Authenticate"], "|", response.json())

`401 Unauthorized` разом із заголовком `WWW-Authenticate: Bearer` — сервер каже клієнтові, **як** автентифікуватися.

### Приклад 3.2. Отримати токени і прочитати їх без ключа

`POST /api/token/` з логіном і паролем → `{"access": …, "refresh": …}`. Токен — три частини через крапку: `header.payload.signature`, кожна в **base64url** (без доповнення `=`). Щоб розкодувати, доповнюємо `=` до довжини, кратної 4.

In [ ]:
def b64url_decode(part):
    """base64url без '=' → байти."""
    return base64.urlsafe_b64decode(part + "=" * (-len(part) % 4))


def b64url_encode(raw):
    """байти → base64url без '='."""
    return base64.urlsafe_b64encode(raw).decode().rstrip("=")


def read_part(part):
    """Частина токена → словник (JSON)."""
    return json.loads(b64url_decode(part))

In [ ]:
tokens = api.post("/api/token/", {"username": "olena", "password": "Sup3r-secret!"}, format="json").json()
access = tokens["access"]
header, payload, signature = access.split(".")
claims = read_part(payload)

print("ключі відповіді:", sorted(tokens))
print("header: ", read_part(header))
print("payload:", claims)
print("живе", claims["exp"] - claims["iat"], "с")
assert claims["user_id"] == str(olena.id) and claims["token_type"] == "access"

Payload **не зашифрований**: ми щойно прочитали `user_id`, тип токена і час життя без жодного ключа. Тому в токен не кладуть нічого секретного. `jti` — унікальний номер токена, `iat`/`exp` — коли видано і до якого моменту (Unix-час) він дійсний.

### Приклад 3.3. Запит з токеном

In [ ]:
api.credentials(HTTP_AUTHORIZATION=f"Bearer {access}")
response = api.get("/api/notes/")
print(response.status_code, [note["title"] for note in response.json()])
api.credentials()                                     # прибрати заголовок для наступних прикладів

`api.credentials(...)` додає заголовок до **кожного** наступного запиту цього клієнта, а `api.credentials()` без аргументів його прибирає. Олена бачить обидві свої нотатки.

### Приклад 3.4. Підпис вручну

Захищає токен третя частина — **підпис**. Для `alg: HS256` це `HMAC-SHA256("header.payload", SECRET_KEY)` у base64url. Порахуємо його стандартною бібліотекою і порівняємо з підписом у токені:

In [ ]:
import hmac


def hs256_signature(header_part, payload_part, key):
    """Підпис JWT з алгоритмом HS256."""
    message = f"{header_part}.{payload_part}".encode()
    return b64url_encode(hmac.new(key.encode(), message, hashlib.sha256).digest())

In [ ]:
print("у токені:  ", signature)
print("порахували:", hs256_signature(header, payload, settings.SECRET_KEY))
print("чужим ключем:", hs256_signature(header, payload, "інший-ключ"))
assert hs256_signature(header, payload, settings.SECRET_KEY) == signature

Підпис збігся, бо ми знаємо `SECRET_KEY`. З іншим ключем — зовсім інший рядок. Сервер, отримавши токен, робить те саме: перераховує HMAC і порівнює. Тому:

- змінити payload і лишити старий підпис — не вийде (наступні приклад і задача);
- хто знає `SECRET_KEY`, той сам випускає «справжні» токени (приклад 4.5).

### Приклад 3.5. Три підробки — три `401`

Перевіримо кожну гілку перевірки на сервері: зіпсований підпис, токен, підписаний чужим ключем, і прострочений токен. `AccessToken.for_user` — так сам simplejwt випускає токен; `set_exp(lifetime=-…)` робить його вже простроченим.

In [ ]:
from datetime import timedelta

import jwt
from rest_framework_simplejwt.tokens import AccessToken


def status_with(token):
    """Статус і текст помилки GET /api/notes/ з цим токеном."""
    client = APIClient()
    client.credentials(HTTP_AUTHORIZATION=f"Bearer {token}")
    response = client.get("/api/notes/")
    detail = "ok" if response.status_code == 200 else response.json()["detail"]
    return response.status_code, detail

In [ ]:
tampered = f"{header}.{payload}.{signature[:-4]}AAAA"
foreign = jwt.encode({**claims, "user_id": str(ivan.id)}, "зовсім-інший-ключ-підпису-довжиною-понад-32-байти",
                     algorithm="HS256")
expired = AccessToken.for_user(olena)
expired.set_exp(lifetime=-timedelta(seconds=1))

for name, token in (("справжній", access), ("підправлений", tampered), ("чужий ключ", foreign), ("прострочений", str(expired))):
    print(f"{name:13} → {status_with(token)}")

Усі три відмови — з тим самим повідомленням: сервер не підказує, **яка** перевірка не пройшла (зайва підказка допомагає зловмисникові).

**🔮 Прогноз:** Тарас скопіював токен Олени, у payload замінив `user_id` на свій і закодував назад. Підпис лишив Оленин. Що відповість сервер?

<details>
<summary>Відповідь</summary>

`401`: підпис рахувався від **старих** header і payload. Сервер перераховує HMAC від нових — і він не збігається. Без `SECRET_KEY` правильний підпис для зміненого payload не зробити.

</details>

### Задача 2. Підробити `user_id`

Перевір прогноз:

1. зроби `forged_claims` — копію `claims`, але з `"user_id"` = id Тараса **рядком** (`str(taras.id)`);
2. закодуй їх у `forged_payload`: `json.dumps(...).encode()` → `b64url_encode` (приклад 3.2);
3. склей токен `header.forged_payload.signature` (`header`, `signature` — з прикладу 3.2) і збережи статус `GET /api/notes/` з ним у `status_forged` (функція `status_with` з прикладу 3.5 повертає пару `(статус, текст)`).

In [ ]:
# YOUR CODE HERE


In [ ]:
assert read_part(forged_payload)["user_id"] == str(taras.id), "у forged_payload має бути user_id Тараса (рядком)"
print("payload підробки:", read_part(forged_payload))
print("статус:", status_forged)
assert status_forged == 401
print("✅ Задача 2 розв'язана")

### Приклад 3.6. Те саме — засобами simplejwt

На сервері перевірку робить клас `AccessToken`: розбирає, перевіряє підпис і `exp`, а при помилці кидає `TokenError`. Ним зручно перевіряти токени в тестах і скриптах:

In [ ]:
from rest_framework_simplejwt.exceptions import TokenError


def check_token(token):
    try:
        return f"дійсний, user_id={AccessToken(token)['user_id']}"
    except TokenError as error:
        return f"TokenError: {error}"

In [ ]:
for name, token in (("справжній", access), ("підправлений", tampered), ("прострочений", str(expired))):
    print(f"{name:13} → {check_token(token)}")

### Приклад 3.7. Refresh: нова пара без пароля

Access живе 5 хвилин. Коли він «згорів», клієнт надсилає refresh-токен на `/api/token/refresh/` і отримує нову пару — пароль удруге вводити не треба. `ROTATE_REFRESH_TOKENS` видає ще й **новий** refresh.

In [ ]:
fresh = api.post("/api/token/refresh/", {"refresh": tokens["refresh"]}, format="json").json()
print("ключі:", sorted(fresh))
print("новий refresh:", fresh["refresh"] != tokens["refresh"])
print("новий access працює:", status_with(fresh["access"]))
print("refresh як access:", status_with(tokens["refresh"]))

Refresh-токен **не** відчиняє API: у payload `token_type: refresh`, а `JWTAuthentication` приймає лише `access`. Його використовують лише на одному ендпоінті й рідко — тому його важче перехопити.

### Приклад 3.8. Що всередині refresh-токена

In [ ]:
refresh_claims = read_part(tokens["refresh"].split(".")[1])
print({key: refresh_claims[key] for key in ("token_type", "user_id")})
print("живе", (refresh_claims["exp"] - refresh_claims["iat"]) / 3600, "год")

Доба замість 5 хвилин — і тип `refresh`. Відкликати вже виданий токен (кнопка «вийти на всіх пристроях») можна лише з додатком `token_blacklist` — у книзі це «Спробуй самостійно».

### Приклад 3.9. Чому `401`, а не `403`: порядок автентифікаторів

Коли запит без облікових даних, DRF відповідає за **першим** класом автентифікації: якщо він уміє заголовок `WWW-Authenticate` (JWT уміє) — `401`, якщо ні (сесія) — `403`. Збудуємо два крихітні view з різним порядком і викличемо їх напряму через `APIRequestFactory`:

In [ ]:
from rest_framework.authentication import SessionAuthentication
from rest_framework.permissions import IsAuthenticated
from rest_framework.response import Response
from rest_framework.test import APIRequestFactory
from rest_framework.views import APIView
from rest_framework_simplejwt.authentication import JWTAuthentication


class SessionFirst(APIView):
    authentication_classes = [SessionAuthentication, JWTAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        return Response({"user": request.user.username})


class JwtFirst(SessionFirst):
    authentication_classes = [JWTAuthentication, SessionAuthentication]

In [ ]:
factory = APIRequestFactory()
for view in (SessionFirst, JwtFirst):
    response = view.as_view()(factory.get("/"))
    print(f"{view.__name__:12} анонім → {response.status_code}, WWW-Authenticate: {response.get('WWW-Authenticate')}")
response = JwtFirst.as_view()(factory.get("/", HTTP_AUTHORIZATION=f"Bearer {access}"))
print("JwtFirst з токеном →", response.status_code, response.data)

Той самий анонімний запит — `403` або `401` залежно від порядку в списку. `401` правильніший для API: клієнт розуміє, що треба отримати (чи оновити) токен.

### Приклад 3.10. Перебір паролів: throttle

`/api/token/` пускає не більше 5 спроб за хвилину з однієї адреси (`ScopedRateThrottle` у `hello_app/auth_api.py`). Підбираємо пароль, поки не отримаємо `429`, а потім пробуємо **правильний**.

**🔮 Прогноз:** яким буде статус правильного пароля після п'яти невдалих спроб?

<details>
<summary>Відповідь</summary>

`429` — throttle рахує **спроби**, а не помилки, і не знає, правильний пароль чи ні. Справжній користувач почекає хвилину, а зловмисник не зможе перебирати швидко.

</details>

In [ ]:
cache.clear()
api = APIClient()
attempts = []
while not attempts or attempts[-1] != 429:
    attempts.append(api.post("/api/token/", {"username": "olena", "password": f"guess-{len(attempts)}"}, format="json").status_code)
right = api.post("/api/token/", {"username": "olena", "password": "Sup3r-secret!"}, format="json")
print("спроби:", attempts)
print("правильний пароль →", right.status_code, "| чекати", right["Retry-After"], "с")
assert attempts == [401] * 5 + [429] and right.status_code == 429
cache.clear()

---
## 4. Налаштування безпеки

`settings.py` бере `SECRET_KEY`, `DEBUG` і `ALLOWED_HOSTS` зі змінних середовища, а для розробки має «навчальні» значення:

```python
SECRET_KEY = os.environ.get("DJANGO_SECRET_KEY", "django-insecure-crispy-notes-dev-key-change-in-production")
DEBUG = os.environ.get("DJANGO_DEBUG", "1") == "1"
ALLOWED_HOSTS = [host for host in os.environ.get("DJANGO_ALLOWED_HOSTS", "").split(",") if host]
```

### Приклад 4.1. `check --deploy`: що ще не готово до сервера

`manage.py check --deploy` — перевірка Django «перед деплоєм». Запустимо її окремим процесом (як у терміналі) і залишимо з виводу лише коди попереджень:

In [ ]:
import re
import secrets


def deploy_warnings(**env):
    """Коди попереджень `manage.py check --deploy` зі змінними середовища env."""
    result = subprocess.run([sys.executable, "manage.py", "check", "--deploy"], capture_output=True, text=True,
                            env={**os.environ, **env})
    return re.findall(r"\(security\.(W\d+)\)", result.stdout + result.stderr)

In [ ]:
print("як у репозиторії:", deploy_warnings())

Що означають коди:

| Код | Про що |
|---|---|
| `W009` | `SECRET_KEY` короткий або з префіксом `django-insecure-` |
| `W018` | `DEBUG = True` |
| `W020` | порожній `ALLOWED_HOSTS` |
| `W004`, `W008`, `W012`, `W016` | HTTPS: HSTS, редирект на https, `Secure` для cookie сесії й CSRF — вмикають на сервері з сертифікатом (урок 50) |

### Приклад 4.2. Ті самі налаштування — зі змінних середовища

Новий ключ генерує `secrets.token_urlsafe(50)`. Передамо ключ, `DEBUG=0` і домен через змінні — код не змінюється:

In [ ]:
server_env = {"DJANGO_SECRET_KEY": secrets.token_urlsafe(50), "DJANGO_DEBUG": "0",
              "DJANGO_ALLOWED_HOSTS": "notes.example.com"}
print("на сервері:", deploy_warnings(**server_env))

`W009`, `W018`, `W020` зникли — лишились лише попередження про HTTPS. Секрет — у середовищі сервера, а не в git.

### Приклад 4.3. Заголовки безпеки у відповіді

Блок налаштувань безпеки перетворюється на заголовки, які сервер додає до **кожної** відповіді. Відкриємо сторінку входу:

In [ ]:
response = Client().get("/accounts/login/")
for name in ("X-Frame-Options", "X-Content-Type-Options", "Referrer-Policy", "Cross-Origin-Opener-Policy"):
    print(f"{name:28} {response.get(name)}")

- `X-Frame-Options: DENY` — сторінку не вставиш в `<iframe>` чужого сайту (clickjacking);
- `X-Content-Type-Options: nosniff` — браузер не «вгадує» тип файлу;
- `Referrer-Policy` і `Cross-Origin-Opener-Policy` — значення, які `SecurityMiddleware` додає сам.

### Приклад 4.4. Cookie сесії: `HttpOnly` і `SameSite`

Увійдемо формою входу (з паролем, як людина) і подивимось на атрибути cookie `sessionid`:

In [ ]:
browser = Client()
response = browser.post("/accounts/login/", {"username": "olena", "password": "Sup3r-secret!"})
cookie = browser.cookies["sessionid"]
print("вхід →", response.status_code, response["Location"])
print("HttpOnly:", bool(cookie["httponly"]), "| SameSite:", cookie["samesite"], "| Secure:", bool(cookie["secure"]))
print("що в cookie:", cookie.value[:12] + "…  (лише випадковий ключ сесії)")

- `HttpOnly` — JavaScript сторінки не прочитає cookie, тож XSS не вкраде вхід;
- `SameSite=Lax` — cookie не йде із запитами, які починає чужий сайт (частина захисту від CSRF);
- `Secure` — поки `False`: cookie лише через HTTPS вмикаємо на сервері (попередження `W012` у прикладі 4.1).

### Приклад 4.5. Знайди помилку: ключ з репозиторію

Проєкт лежить у публічному репозиторії, а на сервері його запустили як є — без `DJANGO_SECRET_KEY`. Хтось прочитав `settings.py` і **сам** підписав токен:

**🔮 Прогноз:** що відповість API на такий токен — без пароля і без `/api/token/`?

<details>
<summary>Відповідь</summary>

`200` і всі нотатки Олени: підпис правильний, бо зроблений тим самим ключем. `exp` = 2100 рік, тож доступ — «назавжди». Рятує лише **новий** ключ у змінній середовища: усі токени, сесії й посилання скидання, підписані старим, стануть недійсними. Ключ, що потрапив у git, вважається скомпрометованим назавжди.

</details>

In [ ]:
print("ключ з репозиторію:", settings.SECRET_KEY[:20] + "…")
forged = jwt.encode({"token_type": "access", "user_id": str(olena.id), "exp": 4102444800, "jti": "x"},
                    settings.SECRET_KEY, algorithm="HS256")
client = APIClient()
client.credentials(HTTP_AUTHORIZATION=f"Bearer {forged}")
response = client.get("/api/notes/")
print("підроблений токен →", response.status_code, [note["title"] for note in response.json()])

---
## 5. OWASP Top 10 на нашому коді

[OWASP Top 10](https://owasp.org/Top10/) — рейтинг найпоширеніших ризиків вебзастосунків. Де вони в застосунку нотаток:

| Ризик | Де в проєкті | Що захищає | Приклад |
|---|---|---|---|
| **A01** Broken Access Control | чужа нотатка; нотатка групи в API | `404` для чужих, `403` для учасників, групи у формі — лише свої | 1.3–1.7 |
| **A02** Cryptographic Failures | паролі, `SECRET_KEY` | PBKDF2 з сіллю; ключ зі змінної середовища | 2.1–2.5, 4.5 |
| **A03** Injection | фільтри й пошук | ORM передає значення параметрами; автоекранування шаблонів | 5.1 |
| **A05** Security Misconfiguration | `DEBUG`, `ALLOWED_HOSTS`, заголовки | `check --deploy`, блок налаштувань безпеки | 4.1–4.4 |
| **A07** Identification and Authentication Failures | вхід, скидання пароля, API | throttle, однакова відповідь для невідомого email, короткий access, валідатори | 2.6–2.8, 3.10 |

### Приклад 5.1. Ін'єкція не проходить через ORM

Класичний рядок SQL-ін'єкції `' OR '1'='1` у пошуку API. Якби пошук склеював SQL рядком, Іван побачив би всі нотатки. ORM передає значення **параметром**:

In [ ]:
attack = "' OR '1'='1"
client = APIClient()
client.force_authenticate(ivan)
print("Іван шукає ін'єкцією →", client.get("/api/notes/", {"search": attack}).json())

sql, params = selectors.get_user_notes(ivan, search=attack).query.sql_with_params()
print("плейсхолдерів %s у SQL:", sql.count("%s"), "| параметри:", params[-2:])

Рядок атаки — лише значення параметра `LIKE '%…%'`, а не частина SQL. Порожній список: нотаток з таким текстом в Івана немає.

---
## 6. Правило автора в одному місці

Знахідка уроку: сайт перевіряв автора, а API — ні. Правило, записане в кількох місцях, колись забудуть. У DRF для цього є **permission-клас** — два методи:

- `has_permission(request, view)` — для **будь-якого** запиту до view (ще до пошуку об'єкта);
- `has_object_permission(request, view, obj)` — для конкретного **об'єкта**, коли view викликає `check_object_permissions`.

### Приклад 6.1. Найпростіший permission: лише читання

`ReadOnly` дозволяє лише безпечні методи (`permissions.SAFE_METHODS`: `GET`, `HEAD`, `OPTIONS`). Щоб перевірити клас без сервера, підставимо замість запиту `SimpleNamespace` з потрібними атрибутами — permission бачить лише `request.method` і `request.user`.

In [ ]:
from types import SimpleNamespace

from rest_framework import permissions


class ReadOnly(permissions.BasePermission):
    message = "Лише читання."

    def has_permission(self, request, view):
        return request.method in permissions.SAFE_METHODS


def fake_request(method, user):
    return SimpleNamespace(method=method, user=user)

In [ ]:
check = ReadOnly()
print(permissions.SAFE_METHODS)
print({method: check.has_permission(fake_request(method, olena), None) for method in ("GET", "HEAD", "PATCH", "DELETE")})

### Приклад 6.2. Permission у справжньому view

Клас вішають на view через `permission_classes`. Збудуємо view, що лише читає, і викличемо його від імені Олени (`force_authenticate` для `APIRequestFactory`):

In [ ]:
from rest_framework.test import force_authenticate


class NotesReadOnly(APIView):
    permission_classes = [IsAuthenticated, ReadOnly]

    def get(self, request):
        return Response([note.title for note in selectors.get_user_notes(request.user)])

    def post(self, request):
        return Response({"created": True}, status=201)

In [ ]:
for method in ("get", "post"):
    request = getattr(factory, method)("/")
    force_authenticate(request, user=olena)
    response = NotesReadOnly.as_view()(request)
    print(method.upper(), "→", response.status_code, response.data if response.status_code == 200 else str(response.data["detail"]))

`POST` зупинився на `has_permission` ще до виклику `post` — з повідомленням з `message`. Усі класи зі списку мають сказати «так».

### Задача 3. `IsAuthorOrReadOnly`

Напиши permission-клас, що замінить `_get_own_note`:

- `message = "Змінювати й видаляти нотатку може лише її автор."`;
- `has_object_permission(self, request, view, obj)`: безпечні методи (`permissions.SAFE_METHODS`) — `True` (видимість уже вирішили селектори); решта — лише якщо `obj.user_id == request.user.id`.

У ViewSet його вішають як `permission_classes = [IsAuthenticated, IsAuthorOrReadOnly]` і викликають `self.check_object_permissions(request, note)` після того, як нотатку знайдено.

In [ ]:
# YOUR CODE HERE


In [ ]:
check = IsAuthorOrReadOnly()
table = {(method, user.username): check.has_object_permission(fake_request(method, user), None, wifi)
         for method in ("GET", "PATCH", "DELETE") for user in (olena, taras)}
print(table)
assert table[("GET", "taras")] is True, "учасник групи має читати нотатку (GET — безпечний метод)"
assert table == {("GET", "olena"): True, ("GET", "taras"): True, ("PATCH", "olena"): True,
                 ("PATCH", "taras"): False, ("DELETE", "olena"): True, ("DELETE", "taras"): False}
assert check.message == "Змінювати й видаляти нотатку може лише її автор."
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. Чим автентифікація відрізняється від авторизації? Що з них дає JWT?
2. Чому учасник групи отримує `403`, а сторонній — `404`?
3. Навіщо сіль і навіщо хешування навмисно повільне?
4. Що в JWT можна прочитати без ключа і чого без ключа зробити не можна?
5. Чому `SECRET_KEY` не можна тримати в публічному репозиторії?
6. Чому посилання скидання пароля перестає діяти після зміни пароля — навіть на той самий?
7. Чому анонімний запит до API отримує `401`, а не `403`?

<details>
<summary>Відповіді</summary>

1. Автентифікація — «хто ти», авторизація — «що тобі можна». JWT (як і сесія) — лише автентифікація; права вирішує код (селектори, перевірка автора, permissions).
2. Учасник і так бачить нотатку — приховувати нічого; сторонньому `403` підтвердив би, що нотатка існує.
3. Сіль робить однакові паролі різними хешами й ламає заздалегідь пораховані таблиці; повільність робить перебір після витоку бази надто довгим (приклад 2.5).
4. Прочитати header і payload (base64url); змінити їх і мати правильний підпис — ні (приклад 3.4, задача 2).
5. Ним підписано JWT, сесії й посилання скидання пароля: хто знає ключ, сам випустить «справжній» токен для будь-якого користувача (приклад 4.5).
6. Токен залежить від хешу пароля, а новий пароль — це нова сіль і новий хеш (приклад 2.8).
7. Першим у списку стоїть `JWTAuthentication`, який має заголовок `WWW-Authenticate` (приклад 3.9).

</details>

## Далі

- Книга курсу: [Урок 41](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_41/) — знахідка «API обходив перевірку автора», скидання пароля, `check --deploy`, OWASP Top 10 на нашому коді.
- У папці `crispy_notes_project`: `python manage.py test`, `python manage.py check --deploy`.
- **Урок 42** — тестування API агрегатора новин.